# Apply Analysis

Analyze a **completed** Terraform apply from **`terraform apply -json`** UI output or **`TF_LOG=json`** trace output.

Reports apply counts, durations, and longest-running resources.

**UI JSON capture** (non-interactive only):

```bash
terraform apply -json -auto-approve > apply-ui.json
export TERRAFORM_LOG_PATH=apply-ui.json
```

**TF_LOG trace capture** (interactive OK):

```bash
export TF_LOG=json
export TF_LOG_PATH=apply-tflog.log
terraform apply
export TERRAFORM_LOG_PATH=apply-tflog.log
```

For hung or partial applies use `apply/hang-analysis.ipynb`. Run `_shared/whatisit.ipynb` if unsure.

In [ ]:
import sys
from pathlib import Path

_nb_root = Path.cwd()
if (_nb_root.parent / "notebook_setup.py").is_file():
    sys.path.insert(0, str(_nb_root.parent))
elif (_nb_root / "notebook_setup.py").is_file():
    sys.path.insert(0, str(_nb_root))

import notebook_setup

notebook_setup.setup()

import pandas as pd
import commonlib.prep_apply_data as prep_apply_data
import commonlib.gencharts as gencharts
import matplotlib.pyplot as plt
import commonlib.config as config

In [ ]:
c = config.Config()
print(c.TERRAFORM_LOG_PATH)
parsed_records = prep_apply_data.read_json_from_file(c.TERRAFORM_LOG_PATH)
normalized_records = prep_apply_data.normalize_records(parsed_records)
df = pd.json_normalize(normalized_records)

if df.empty:
    raise ValueError(
        "No apply records found in TF_LOG=json trace or terraform apply -json UI output. "
        "Confirm the file with _shared/whatisit.ipynb."
    )

print(sorted(df["type"].drop_duplicates().tolist()))

In [ ]:
# Match apply_start to apply_complete per resource, preserving order for repeated applies
starts = (
    df[df["type"] == "apply_start"]
    .copy()
    .sort_values(["resource", "timestamp"])
)
starts["start_timestamp"] = pd.to_datetime(starts["timestamp"], utc=True, errors="coerce")
starts["run"] = starts.groupby("resource").cumcount() + 1
starts = starts[["resource", "run", "start_timestamp", "resource_type", "resource_name", "action"]]

ends = (
    df[df["type"] == "apply_complete"]
    .copy()
    .sort_values(["resource", "timestamp"])
)
ends["end_timestamp"] = pd.to_datetime(ends["timestamp"], utc=True, errors="coerce")
ends["run"] = ends.groupby("resource").cumcount() + 1
ends = ends[["resource", "run", "end_timestamp", "elapsed_seconds"]]

df_merged_apply = starts.merge(ends, on=["resource", "run"], how="outer")
df_merged_apply["time_diff_minutes"] = (
    (df_merged_apply["end_timestamp"] - df_merged_apply["start_timestamp"])
    .dt.total_seconds()
    / 60
)

## Type Analysis

In [ ]:
gencharts.generate_plt_by_resource_type(df, "apply_start", top_n=10)
gencharts.generate_plt_by_resource_type(df, "apply_complete", top_n=10)
gencharts.generate_plt_by_resource_type(df, "apply_progress", top_n=10)
gencharts.generate_plt_by_resource_type(df, "apply_errored", top_n=10)

## Duration Analysis

In [ ]:
if not df_merged_apply.dropna(subset=["time_diff_minutes"]).empty:
    gencharts.generate_duration_by_resource_type(df_merged_apply, metric="total", top_n=10)
    gencharts.generate_duration_by_resource_type(df_merged_apply, metric="average", top_n=10)
else:
    print("No matched apply_start/apply_complete pairs to chart.")

## Longest Running Applies

In [ ]:
if df_merged_apply.dropna(subset=["time_diff_minutes"]).empty:
    print("No matched apply_start/apply_complete pairs to list.")
else:
    display(
        df_merged_apply[
            ["resource", "resource_type", "action", "start_timestamp", "end_timestamp", "time_diff_minutes", "run"]
        ]
        .copy()
        .sort_values(by="time_diff_minutes", ascending=False)
        .head(20)
    )